# Run a Workplan with C-star CLI

This example demonstrates how to launch a workplan in C-Star. A workplan can call any number of jobs (e.g. ROMS-MARBL blueprints). These jobs can depend on a previous job to finish before beginning.

The jobs in this workplan run regional ocean model simulations, including biogeochemical processes, defined by ROMS-MARBL blueprints. You may want to refer to the [Terminology and Concepts page](../terminology.rst) for a broader overview of what a workplan is and how it fits into the bigger picture.

## Prerequisities

Similar to the Blueprint tutorial [prerequisites](./tutorial_bp.ipynb), in order to run this notebook, you will first need to:

- [Install](../installation.rst) `C-Star`
- Activate the environment in your terminal with `conda activate cstar_env`
- If you want to run this example notebook, first create an `ipykernel` by running this command in your terminal:

  `python -m ipykernel install --user --name cstar_env --display-name cstar_env`

- Launch this notebook from your terminal with `jupyter lab docs/tutorials/tutorial_wrkpln.ipynb` and selected the `cstar_env` in the kernel dropdown within the notebook.

<div class="alert alert-info">

Note

This short example is runnable on a MacOS laptop with the default C-Star configuration settings. If you run this example or another use case on an HPC, please visit the  [configuration](../configuration.rst) to learn how to integrate C-Star with SLURM and configure other important behaviors.
</div>

To run each of the ROMS-MARBL applications (jobs), the following files first need to be created specifically for the simulations you'd like to run:
* Input data files (e.g. surface and boundary conditions, initial conditions, grid definitions, etc., typically as `.nc` files)
* Compile-time files (`.opt`) that define parameters and toggle options in the ROMS code
* Runtime files (`.in`, `marbl_*`) files that specify additional settings

For this example, we provide blueprints along with all files needed to run each simulation.\
The blueprints in this example ([wales](wales_toy_blueprint.yaml) and [wio](wio_toy_blueprint.yaml) YAMLs) reside in the same directory as this notebook, and can also be viewed alongside the data they point to in our [examples repository](https://github.com/CWorthy-ocean/cstar_blueprint_roms_marbl_example).

To create your own set of simulations, consider using this workplan as a template and creating your own blueprints along with their supporting files. [Roms-Tools](https://roms-tools.readthedocs.io/en/latest/index.html) can be used for input data files, or [C-SON Forge](https://cworthy-ocean.github.io/cson-forge/installation/) for all necessary files (input, compile-time, run-time, and blueprint).

## Inspect and run the simulation

<div class="alert alert-info">

Note

`C-Star` is intended to be run as a command line utility. We have formulated this tutorial as a Jupyter Notebook to easily step through commands and see results as they run, using the `!` Jupyter directive to run terminal and CLI commands. All of these commands could be similarly run from a terminal shell without the `!`.
</div>

### Define the workplan

We can print the content of the workplan with `cat wrkpln.yaml`, and see that each of the attributes described in the [workplans page](../workpans.rst) are defined.

The workplan below lists two jobs to be run.
- Job 1 is called `wio` and runs the ROMS-MARBL model (or, application) on the blueprint provided (`wio_toy_blueprint.yaml`) - a domain of the Western Indian Ocean.\
- Job 2, called `wales` also runs the ROMS-MARBL application, but calls the `wales_toy_blueprint.yaml` (also provided) - this runs a domain of Wales.

See the [blueprint tutuorial](./tutorial_bp.ipynb) for further explanation of the `wio` domain. Both domains along with their necessary files can be found in our [examples repository](https://github.com/CWorthy-ocean/cstar_blueprint_roms_marbl_example/).

**Print out the workplan contents:**

In [2]:
!cat wrkplan.yaml

name: Run 2 jobs
description: Run two domains using ROMS-MARBL simulations
state: draft

steps:
- name: wio
  application: roms_marbl
  blueprint: /Users/sam.maticka/workdir/C-Star/docs/tutorials/wio_toy_blueprint.yaml
- name: wales
  application: roms_marbl
  blueprint: /Users/sam.maticka/workdir/C-Star/docs/tutorials/wales_toy_blueprint.yaml
  depends_on:
  - wio


### Job dependence

Note how the `wales` job has a dependence on the `wio` job. This tells `C-Star` that the `wio` job must finish before the `wales` job can start. For this workplan, we use two separate domains that depend on separate files, so a dependence is not necessary here, it is just for purpose of demonstration.\
The `depends_on` paramter may be used when the results of one simulation (job 1) are necessary to start the following similution (job 2) (e.g. job 2 input files point to the output files from job 1). 

## Check the workplan for syntax

Similar to a blueprint, once the workplan is complete, we can use `C-Star` to proofread it. The `cstar workplan` command has a functionality `check` to confirm that the syntax of the workplan YAML is correct and all necessary parameters are included.\
This does not check for things such as whether the input files exist, it simply checks that the workplan is syntactically correct.

In [3]:
!cstar workplan check wrkplan.yaml

The workplan is valid


## Run the model from the blueprint

In order to run the workplan, a `run-id` needs to be prescribed. This run-id is stored, so if you wnat to rerun the simulation starting from scratch (clearing.. NEED TO FINISH

In [5]:
!cstar workplan run --run-id test_001 wrkplan.yaml

2026-02-24 11:43:10,827 [INFO] - dag_runner.py:179 - A time-split workplan will be executed.
Local run of `roms_marbl` created pid: 17235
2026-02-24 11:43:11,048 [INFO] - orchestration.py:544 - Launched step: wio
2026-02-24 11:43:46,314 [ERROR] - worker.py:468 - An unexpected exception occurred during the simulation
Traceback (most recent call last):
  File "/opt/miniconda3/envs/cstar_env/lib/python3.13/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/opt/miniconda3/envs/cstar_env/lib/python3.13/site-packages/urllib3/connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
  File "/opt/miniconda3/envs/cstar_env/lib/python3.13/http/client.py", line 1450, in getresponse
    response.begin()
    ~~~~~~~~~~~~~~^^
  File "/opt/miniconda3/envs/cstar_env/lib/python3.13/http/client.py", line 336, in begin
    version, status, reason = self._read_status()
                              ~~~~~~~~~~~~~~~~~^^